# 04 — Amélioration des features

Ce notebook compare **trois approches** d'amélioration sur les modèles classiques (LR, RF, XGBoost) :

| Approche | Description |
|---|---|
| **Baseline** | TF-IDF (1-2grammes, min_df=3) + 4 métadonnées |
| **Extended** | Baseline + 17 nouvelles features (sujet, parti, linguistiques) |
| **Piste 1** | TF-IDF enrichi avec le sujet concaténé au texte |
| **Piste 2** | TF-IDF avec min_df=2 (vocabulaire élargi) |


## 0. Imports

In [16]:
import sys
import os
import warnings
warnings.filterwarnings("ignore")

sys.path.insert(0, "src")

import pandas as pd
import numpy as np

from preprocessing import load_liar, preprocess_liar
from features import TfidfFeatures, CombinedFeatures, META_COLS, META_COLS_EXTENDED
from train import train_logistic_regression, train_random_forest, train_xgboost
from evaluate import evaluate_model, save_results

## 1. Chargement & preprocessing

In [17]:
train_raw, valid_raw, test_raw = load_liar("data/raw")
train = preprocess_liar(train_raw)
valid = preprocess_liar(valid_raw)
test  = preprocess_liar(test_raw)

y_train = train["label_encoded"].values
y_test  = test["label_encoded"].values

LIAR chargé — train: 10240, valid: 1284, test: 1267


In [18]:
# Distribution des features linguistiques par classe
print("Moyenne par classe (train) :")
for col in ["has_absolute_lang", "has_hedging", "stmt_word_count"]:
    means = train.groupby("label_3class")[col].mean().round(3)
    print(f"  {col:<22}: fake={means.get('fake',0):.3f}  "
          f"nuanced={means.get('nuanced',0):.3f}  real={means.get('real',0):.3f}")

Moyenne par classe (train) :
  has_absolute_lang     : fake=0.210  nuanced=0.198  real=0.263
  has_hedging           : fake=0.089  nuanced=0.096  real=0.109
  stmt_word_count       : fake=17.029  nuanced=18.515  real=18.251


## 2. Baseline

In [19]:
tfidf_base = TfidfFeatures(max_features=20_000, ngram_range=(1, 2))
tfidf_base.fit_transform(train["statement_clean"])

combined_base = CombinedFeatures(text_features=tfidf_base, meta_cols=META_COLS)
X_train_base = combined_base.fit_transform(train)
X_test_base  = combined_base.transform(test)

print(f"Baseline : X_train shape = {X_train_base.shape}")

TF-IDF fit — vocabulaire : 9049 features, 10240 documents
Baseline : X_train shape = (10240, 9053)


In [20]:
results_baseline = []

lr_base = train_logistic_regression(X_train_base, y_train)
results_baseline.append(evaluate_model(y_test, lr_base.predict(X_test_base), "LR — baseline"))

rf_base = train_random_forest(X_train_base, y_train, n_estimators=300)
results_baseline.append(evaluate_model(y_test, rf_base.predict(X_test_base), "RF — baseline"))

xgb_base = train_xgboost(X_train_base, y_train)
X_test_base_dense = X_test_base.toarray() if hasattr(X_test_base, "toarray") else X_test_base
results_baseline.append(evaluate_model(y_test, xgb_base.predict(X_test_base_dense), "XGBoost — baseline"))

Entraînement Logistic Regression...
  LR — terminé

  LR — baseline
  Accuracy    : 0.5375
  F1 macro    : 0.5442   ← métrique principale
  F1 weighted : 0.5338

  F1 par classe :
    fake       : 0.6465
    nuanced    : 0.4705
    real       : 0.5156

              precision    recall  f1-score   support

        fake       0.61      0.69      0.65       341
     nuanced       0.51      0.43      0.47       477
        real       0.50      0.53      0.52       449

    accuracy                           0.54      1267
   macro avg       0.54      0.55      0.54      1267
weighted avg       0.53      0.54      0.53      1267

Entraînement Random Forest...
  RF — terminé

  RF — baseline
  Accuracy    : 0.5651
  F1 macro    : 0.5697   ← métrique principale
  F1 weighted : 0.5616

  F1 par classe :
    fake       : 0.6431
    nuanced    : 0.4874
    real       : 0.5786

              precision    recall  f1-score   support

        fake       0.62      0.67      0.64       341
     nuanc

## 3. Extended features

Ajout de 17 features aux métadonnées : longueur du texte, langage absolu/nuancé, encodage one-hot du parti, indicateurs binaires par sujet politique.

In [21]:
tfidf_ext = TfidfFeatures(max_features=20_000, ngram_range=(1, 2))
tfidf_ext.fit_transform(train["statement_clean"])

combined_ext = CombinedFeatures(text_features=tfidf_ext, meta_cols=META_COLS_EXTENDED)
X_train_ext = combined_ext.fit_transform(train)
X_test_ext  = combined_ext.transform(test)

print(f"Extended : X_train shape = {X_train_ext.shape} (+{X_train_ext.shape[1] - X_train_base.shape[1]} features)")

TF-IDF fit — vocabulaire : 9049 features, 10240 documents
Extended : X_train shape = (10240, 9070) (+17 features)


In [22]:
results_extended = []

lr_ext = train_logistic_regression(X_train_ext, y_train)
results_extended.append(evaluate_model(y_test, lr_ext.predict(X_test_ext), "LR — extended"))

rf_ext = train_random_forest(X_train_ext, y_train, n_estimators=300)
results_extended.append(evaluate_model(y_test, rf_ext.predict(X_test_ext), "RF — extended"))

xgb_ext = train_xgboost(X_train_ext, y_train)
X_test_ext_dense = X_test_ext.toarray() if hasattr(X_test_ext, "toarray") else X_test_ext
results_extended.append(evaluate_model(y_test, xgb_ext.predict(X_test_ext_dense), "XGBoost — extended"))

Entraînement Logistic Regression...
  LR — terminé

  LR — extended
  Accuracy    : 0.5375
  F1 macro    : 0.5437   ← métrique principale
  F1 weighted : 0.5338

  F1 par classe :
    fake       : 0.6390
    nuanced    : 0.4658
    real       : 0.5262

              precision    recall  f1-score   support

        fake       0.60      0.68      0.64       341
     nuanced       0.51      0.43      0.47       477
        real       0.51      0.55      0.53       449

    accuracy                           0.54      1267
   macro avg       0.54      0.55      0.54      1267
weighted avg       0.53      0.54      0.53      1267

Entraînement Random Forest...
  RF — terminé

  RF — extended
  Accuracy    : 0.5635
  F1 macro    : 0.5677   ← métrique principale
  F1 weighted : 0.5605

  F1 par classe :
    fake       : 0.6325
    nuanced    : 0.4926
    real       : 0.5780

              precision    recall  f1-score   support

        fake       0.61      0.65      0.63       341
     nuanc

## 4. Piste 1 — Subject intégré dans le TF-IDF

Au lieu de créer des indicateurs binaires par sujet, on **concatène les tags de sujet au texte nettoyé** avant la vectorisation TF-IDF.  
Ainsi `"health-care taxes"` devient des tokens à part entière : le modèle peut apprendre les co-occurrences sujet×mots directement dans l'espace vectoriel.

> Exemple : `statement_clean = "obama plan cut spending"` + `subject = "health-care,taxes"`  
> → texte enrichi : `"obama plan cut spending healthrcare taxes"`

In [23]:
def add_subject_to_text(df: pd.DataFrame) -> pd.Series:
    """
    Concatène les tags de sujet au texte nettoyé.
    Les tirets sont remplacés par des underscores pour former
    des tokens cohérents (health-care -> health_care).
    """
    subjects = (
        df["subject"]
        .fillna("")
        .str.lower()
        .str.replace("-", "_", regex=False)
        .str.replace(",", " ", regex=False)
    )
    return df["statement_clean"] + " " + subjects

train_text_p1 = add_subject_to_text(train)
test_text_p1  = add_subject_to_text(test)

print("Exemple de texte enrichi :")
print(" Avant:", train["statement_clean"].iloc[0])
print(" Après:", train_text_p1.iloc[0])

Exemple de texte enrichi :
 Avant: annies list political group supports third trimester abortions demand
 Après: annies list political group supports third trimester abortions demand abortion


In [24]:
tfidf_p1 = TfidfFeatures(max_features=20_000, ngram_range=(1, 2))
tfidf_p1.fit_transform(train_text_p1)

combined_p1 = CombinedFeatures(text_features=tfidf_p1, meta_cols=META_COLS)

# On doit passer le texte enrichi manuellement via statement_clean
train_p1 = train.copy()
train_p1["statement_clean"] = train_text_p1
test_p1  = test.copy()
test_p1["statement_clean"]  = test_text_p1

X_train_p1 = combined_p1.fit_transform(train_p1)
X_test_p1  = combined_p1.transform(test_p1)

print(f"Piste 1 : X_train shape = {X_train_p1.shape}")

TF-IDF fit — vocabulaire : 9958 features, 10240 documents
Piste 1 : X_train shape = (10240, 9962)


In [25]:
results_p1 = []

lr_p1 = train_logistic_regression(X_train_p1, y_train)
results_p1.append(evaluate_model(y_test, lr_p1.predict(X_test_p1), "LR — piste1 (subj text)"))

rf_p1 = train_random_forest(X_train_p1, y_train, n_estimators=300)
results_p1.append(evaluate_model(y_test, rf_p1.predict(X_test_p1), "RF — piste1 (subj text)"))

xgb_p1 = train_xgboost(X_train_p1, y_train)
X_test_p1_dense = X_test_p1.toarray() if hasattr(X_test_p1, "toarray") else X_test_p1
results_p1.append(evaluate_model(y_test, xgb_p1.predict(X_test_p1_dense), "XGBoost — piste1 (subj text)"))

Entraînement Logistic Regression...
  LR — terminé

  LR — piste1 (subj text)
  Accuracy    : 0.5304
  F1 macro    : 0.5368   ← métrique principale
  F1 weighted : 0.5259

  F1 par classe :
    fake       : 0.6440
    nuanced    : 0.4561
    real       : 0.5103

              precision    recall  f1-score   support

        fake       0.60      0.70      0.64       341
     nuanced       0.50      0.42      0.46       477
        real       0.50      0.52      0.51       449

    accuracy                           0.53      1267
   macro avg       0.53      0.55      0.54      1267
weighted avg       0.53      0.53      0.53      1267

Entraînement Random Forest...
  RF — terminé

  RF — piste1 (subj text)
  Accuracy    : 0.5588
  F1 macro    : 0.5642   ← métrique principale
  F1 weighted : 0.5554

  F1 par classe :
    fake       : 0.6461
    nuanced    : 0.4813
    real       : 0.5654

              precision    recall  f1-score   support

        fake       0.62      0.67      0.65 

## 5. Piste 2 — Vocabulaire élargi (min_df = 2)

Avec `min_df=3`, le vocabulaire TF-IDF est plafonné à ~9 000 features alors que `max_features=20 000` permettrait bien plus.  
En abaissant à `min_df=2`, on capture les termes qui apparaissent dans **au moins 2 documents** : noms propres politiques, termes techniques spécifiques à certains sujets.

Risque : plus de bruit (hapax et quasi-hapax), mais potentiellement plus de signal discriminant.

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import StandardScaler

# On crée un TF-IDF avec min_df=2 directement
vectorizer_p2 = TfidfVectorizer(
    max_features=20_000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,                          # <-- changement clé
    token_pattern=r"\b[a-zA-Z']{2,}\b",
)

X_tfidf_train_p2 = vectorizer_p2.fit_transform(train["statement_clean"].fillna(""))
X_tfidf_test_p2  = vectorizer_p2.transform(test["statement_clean"].fillna(""))

print(f"Vocabulaire min_df=3 : {X_train_base.shape[1] - 4} features TF-IDF")
print(f"Vocabulaire min_df=2 : {X_tfidf_train_p2.shape[1]} features TF-IDF")
print(f"  Gain : +{X_tfidf_train_p2.shape[1] - (X_train_base.shape[1] - 4)} tokens")

Vocabulaire min_df=3 : 9049 features TF-IDF
Vocabulaire min_df=2 : 16467 features TF-IDF
  Gain : +7418 tokens


In [27]:
# Combinaison avec les métadonnées (même scaler que baseline)
scaler_p2 = StandardScaler(with_mean=False)
X_meta_train = train[META_COLS].fillna(0).values.astype("float32")
X_meta_test  = test[META_COLS].fillna(0).values.astype("float32")
X_meta_train_scaled = scaler_p2.fit_transform(X_meta_train)
X_meta_test_scaled  = scaler_p2.transform(X_meta_test)

X_train_p2 = hstack([X_tfidf_train_p2, csr_matrix(X_meta_train_scaled)])
X_test_p2  = hstack([X_tfidf_test_p2,  csr_matrix(X_meta_test_scaled)])

print(f"Piste 2 : X_train shape = {X_train_p2.shape}")

Piste 2 : X_train shape = (10240, 16471)


In [28]:
results_p2 = []

lr_p2 = train_logistic_regression(X_train_p2, y_train)
results_p2.append(evaluate_model(y_test, lr_p2.predict(X_test_p2), "LR — piste2 (min_df=2)"))

rf_p2 = train_random_forest(X_train_p2, y_train, n_estimators=300)
results_p2.append(evaluate_model(y_test, rf_p2.predict(X_test_p2), "RF — piste2 (min_df=2)"))

xgb_p2 = train_xgboost(X_train_p2, y_train)
X_test_p2_dense = X_test_p2.toarray() if hasattr(X_test_p2, "toarray") else X_test_p2
results_p2.append(evaluate_model(y_test, xgb_p2.predict(X_test_p2_dense), "XGBoost — piste2 (min_df=2)"))

Entraînement Logistic Regression...
  LR — terminé

  LR — piste2 (min_df=2)
  Accuracy    : 0.5375
  F1 macro    : 0.5454   ← métrique principale
  F1 weighted : 0.5346

  F1 par classe :
    fake       : 0.6537
    nuanced    : 0.4727
    real       : 0.5098

              precision    recall  f1-score   support

        fake       0.62      0.69      0.65       341
     nuanced       0.50      0.44      0.47       477
        real       0.50      0.52      0.51       449

    accuracy                           0.54      1267
   macro avg       0.54      0.55      0.55      1267
weighted avg       0.53      0.54      0.53      1267

Entraînement Random Forest...
  RF — terminé

  RF — piste2 (min_df=2)
  Accuracy    : 0.5556
  F1 macro    : 0.5603   ← métrique principale
  F1 weighted : 0.5515

  F1 par classe :
    fake       : 0.6402
    nuanced    : 0.4744
    real       : 0.5661

              precision    recall  f1-score   support

        fake       0.62      0.66      0.64   

## 6. Comparaison globale

In [29]:
all_results = results_baseline + results_extended + results_p1 + results_p2
df_compare  = pd.DataFrame(all_results).sort_values("f1_macro", ascending=False).reset_index(drop=True)
df_compare

,model,accuracy,f1_macro,f1_weighted,f1_fake,f1_nuanced,f1_real
0,RF — baseline,0.5651,0.5697,0.5616,0.6431,0.4874,0.5786
1,RF — extended,0.5635,0.5677,0.5605,0.6325,0.4926,0.5780
2,RF — piste1 (subj text),0.5588,0.5642,0.5554,0.6461,0.4813,0.5654
3,RF — piste2 (min_df=2),0.5556,0.5603,0.5515,0.6402,0.4744,0.5661
4,XGBoost — baseline,0.5556,0.5577,0.5504,0.6239,0.4824,0.5668
5,XGBoost — piste1 (subj text),0.5501,0.5520,0.5435,0.6304,0.4672,0.5584
6,LR — piste2 (min_df=2),0.5375,0.5454,0.5346,0.6537,0.4727,0.5098
7,LR — baseline,0.5375,0.5442,0.5338,0.6465,0.4705,0.5156
8,LR — extended,0.5375,0.5437,0.5338,0.6390,0.4658,0.5262
9,XGBoost — extended,0.5406,0.5431,0.5348,0.6166,0.4507,0.5620


In [30]:
# Delta F1 macro de chaque piste par rapport à la baseline, pour chaque modèle
approches = {
    "Extended"        : results_extended,
    "Piste 1 (subj)"  : results_p1,
    "Piste 2 (min_df)": results_p2,
}

print(f"{'Modèle':<10} | {'Extended':>12} | {'Piste 1 (subj)':>14} | {'Piste 2 (min_df)':>16}")
print("-" * 60)
for m in ["LR", "RF", "XGBoost"]:
    base = next(r["f1_macro"] for r in results_baseline if r["model"].startswith(m))
    row  = f"{m:<10} |"
    for label, res in approches.items():
        val   = next(r["f1_macro"] for r in res if r["model"].startswith(m))
        delta = val - base
        sign  = "+" if delta >= 0 else ""
        row  += f"  {sign}{delta*100:+.2f} pp     |"
    print(row)

Modèle     |     Extended | Piste 1 (subj) | Piste 2 (min_df)
------------------------------------------------------------
LR         |  -0.05 pp     |  -0.74 pp     |  ++0.12 pp     |
RF         |  -0.20 pp     |  -0.55 pp     |  -0.94 pp     |
XGBoost    |  -1.46 pp     |  -0.57 pp     |  -2.39 pp     |


In [31]:
save_results(all_results, "outputs/results_all_approaches.csv")
print("Résultats sauvegardés -> outputs/results_all_approaches.csv")

Résultats sauvegardés -> outputs/results_all_approaches.csv
Résultats sauvegardés -> outputs/results_all_approaches.csv
